# Experiment: CV HW1 Collaborative Scaffold

Objective:
- Complete homework Step 1-4 and the MLP part of Step 5 in a reusable way.
- Leave shared training and evaluation hooks so teammates can continue with CNN, ResNet, and ViT.


## Collaboration Notes

- Keep shared logic in `hw1_framework.py` so notebook cells stay short and easy to merge.
- Use the `RUN_*` flags before launching long experiments.
- Teammates should reuse the shared data split and training wrapper instead of writing separate loops.


In [ ]:
# Core notebook imports.
from pathlib import Path
import copy

import pandas as pd

import hw1_framework as hw


In [ ]:
# Resolve the project root and create shared output folders.
PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name != "CV_HW1":
    PROJECT_DIR = Path("D:/PythonProjects/CV_HW1")

project_paths = hw.ensure_project_dirs(PROJECT_DIR)
runtime_config = hw.get_default_runtime_config(PROJECT_DIR)

# Fix the seed once so everyone uses the same dataset split and training randomness.
hw.set_seed(runtime_config["seed"])
device = hw.get_device()

print(f"Project directory: {PROJECT_DIR}")
print(f"Device: {device}")
print(f"Output folders: {project_paths}")


## Step 1-4: Data Loading, Split, And Visualization

This section covers the reusable part of the homework that all four models share.


In [ ]:
# Load EMNIST Balanced with one fixed train/validation/test setup.
loaders = hw.load_emnist_balanced(
    data_dir=runtime_config["data_dir"],
    batch_size=runtime_config["batch_size"],
    valid_ratio=runtime_config["valid_ratio"],
    num_workers=runtime_config["num_workers"],
    subset_ratio=1.0,
    augment=runtime_config["augment"],
    rotation_deg=runtime_config["rotation_deg"],
    noise_std=runtime_config["noise_std"],
    blur=runtime_config["blur"],
    seed=runtime_config["seed"],
)

class_names = loaders["class_names"]
dataset_stats = hw.show_dataset_stats(
    loaders["train_dataset"],
    loaders["valid_dataset"],
    loaders["test_dataset"],
    class_names,
)
dataset_stats


In [ ]:
# Plot a few samples so the team can verify label coverage and image orientation early.
sample_figure = hw.show_sample_images(
    dataset=loaders["train_dataset"],
    class_names=class_names,
    num_samples=12,
    seed=runtime_config["seed"],
)
sample_figure


## MLP Baseline And Search Plan

The workflow below is the template that the other three model owners can reuse later.


In [ ]:
# Start from one stable baseline, then replace one factor at a time.
mlp_config = hw.get_default_mlp_config()
mlp_search_plan = {
    "scheduler": ["StepLR", "CosineAnnealingLR", "ReduceLROnPlateau"],
    "activation": ["relu", "leaky_relu", "gelu"],
    "optimizer": ["adam", "sgd", "rmsprop"],
    "normalization": ["batchnorm", "layernorm", "none"],
    "dropout": [0.0, 0.3, 0.5],
}

# L1 and L2 change two fields together, so they are tracked separately from the single-factor loop.
mlp_regularization_trials = [
    {"name": "no_regularization", "l1_lambda": 0.0, "weight_decay": 0.0},
    {"name": "l1_regularization", "l1_lambda": 1e-6, "weight_decay": 0.0},
    {"name": "l2_regularization", "l1_lambda": 0.0, "weight_decay": 1e-4},
]

pd.Series(mlp_config, name="baseline_value")


In [ ]:
# Preview the MLP structure before training and record its parameter count for the report.
mlp_preview = hw.build_mlp(mlp_config)
print(mlp_preview)
print(f"Trainable parameters: {hw.count_parameters(mlp_preview):,}")


In [ ]:
# Keep baseline training opt-in so the notebook is safe to open during collaboration.
RUN_MLP_BASELINE = False

if RUN_MLP_BASELINE:
    mlp_baseline_result = hw.run_training_experiment(
        model_name="mlp_baseline",
        model_builder=hw.build_mlp,
        config=mlp_config,
        loaders=loaders,
        device=device,
        output_dir=project_paths["models"],
    )
    mlp_baseline_curves = hw.plot_training_curves(
        mlp_baseline_result["history"],
        "MLP Baseline",
    )
    mlp_baseline_summary = pd.DataFrame([mlp_baseline_result["summary"]])
    mlp_baseline_summary
else:
    print("Set RUN_MLP_BASELINE = True to train the baseline model.")


In [ ]:
# Sequential search keeps the best setting from one factor before moving to the next factor.
RUN_MLP_SEARCH = False

if RUN_MLP_SEARCH:
    search_tables = []
    tuned_mlp_config = copy.deepcopy(mlp_config)

    for factor_name, candidate_values in mlp_search_plan.items():
        factor_result = hw.run_single_factor_search(
            model_name="mlp",
            model_builder=hw.build_mlp,
            base_config=tuned_mlp_config,
            factor_name=factor_name,
            candidate_values=candidate_values,
            loaders=loaders,
            device=device,
            output_dir=project_paths["models"] / "search",
        )
        search_tables.append(factor_result["results"])
        tuned_mlp_config = factor_result["best_config"]

    regularization_rows = []
    best_regularized_config = copy.deepcopy(tuned_mlp_config)
    best_regularization_score = float("-inf")
    for trial in mlp_regularization_trials:
        trial_config = copy.deepcopy(tuned_mlp_config)
        trial_config["l1_lambda"] = trial["l1_lambda"]
        trial_config["weight_decay"] = trial["weight_decay"]
        trial_result = hw.run_training_experiment(
            model_name=f"mlp_{trial['name']}",
            model_builder=hw.build_mlp,
            config=trial_config,
            loaders=loaders,
            device=device,
            output_dir=project_paths["models"] / "search",
        )
        regularization_rows.append(
            {
                "factor": "regularization",
                "candidate": trial["name"],
                "best_valid_accuracy": trial_result["summary"]["best_valid_accuracy"],
                "best_epoch": trial_result["summary"]["best_epoch"],
                "training_time_sec": trial_result["summary"]["training_time_sec"],
            }
        )
        if trial_result["summary"]["best_valid_accuracy"] > best_regularization_score:
            best_regularization_score = trial_result["summary"]["best_valid_accuracy"]
            best_regularized_config = trial_config

    tuned_mlp_config = best_regularized_config
    mlp_search_results = pd.concat(
        [*search_tables, pd.DataFrame(regularization_rows)],
        ignore_index=True,
    )
    mlp_search_results
else:
    tuned_mlp_config = copy.deepcopy(mlp_config)
    print("Set RUN_MLP_SEARCH = True to launch the MLP factor-by-factor search.")


In [ ]:
# Train the final MLP using the selected config, then evaluate on the test set.
RUN_MLP_FINAL = False

if RUN_MLP_FINAL:
    mlp_final_result = hw.run_training_experiment(
        model_name="mlp_best",
        model_builder=hw.build_mlp,
        config=tuned_mlp_config,
        loaders=loaders,
        device=device,
        output_dir=project_paths["models"],
    )
    mlp_final_curves = hw.plot_training_curves(
        mlp_final_result["history"],
        "MLP Best Model",
    )
    mlp_test_metrics = hw.evaluate_on_test(
        mlp_final_result["model"],
        loaders["test_loader"],
        device,
    )
    mlp_metric_table = pd.DataFrame([hw.summarize_metrics(mlp_test_metrics)])
    mlp_metric_table
else:
    print("Set RUN_MLP_FINAL = True after selecting the final MLP config.")


In [ ]:
# Use the shared visualization hooks once the final MLP checkpoint is ready.
if "mlp_final_result" in globals():
    prediction_figure = hw.preview_predictions(
        model=mlp_final_result["model"],
        loader=loaders["test_loader"],
        class_names=class_names,
        device=device,
        num_samples=6,
    )
    confusion_figure = hw.plot_confusion_matrix_from_preds(
        y_true=mlp_test_metrics["y_true"],
        y_pred=mlp_test_metrics["y_pred"],
        class_names=class_names,
        model_name="MLP",
    )
    prediction_figure
    confusion_figure
else:
    print("Train or load the final MLP model first, then rerun this cell.")


In [ ]:
# Reuse the same best MLP config for the required 30% / 50% / 100% small-sample comparison.
RUN_MLP_SMALL_SAMPLE = False

if RUN_MLP_SMALL_SAMPLE:
    mlp_small_sample_df, mlp_small_sample_runs = hw.run_small_sample_experiment(
        model_name="mlp",
        model_builder=hw.build_mlp,
        base_config=tuned_mlp_config,
        runtime_config=runtime_config,
        sample_ratios=[0.3, 0.5, 1.0],
        device=device,
        output_dir=project_paths["models"],
    )
    mlp_small_sample_df
else:
    print("Set RUN_MLP_SMALL_SAMPLE = True to run the 30% / 50% / 100% comparison.")


## Scaffolds For The Other Three Models

These configs and builders are ready for the teammates who own CNN, ResNet, and ViT.


In [ ]:
# Every teammate should start here and then tune only their own model-specific config.
cnn_config = hw.get_default_cnn_config()
resnet_config = hw.get_default_resnet_config()
vit_config = hw.get_default_vit_config()

scaffold_rows = []
for model_name, config in {
    "cnn": cnn_config,
    "resnet": resnet_config,
    "vit": vit_config,
}.items():
    model = hw.MODEL_BUILDERS[model_name](config)
    scaffold_rows.append(
        {
            "model": model_name,
            "trainable_parameters": hw.count_parameters(model),
            "config_ready": True,
        }
    )

pd.DataFrame(scaffold_rows)


## Shared Step 6 Hook

Once all four best checkpoints are ready, use one shared evaluation cell to keep the comparison fair.


In [ ]:
def evaluate_saved_models(best_model_paths, model_configs, shared_loaders, class_names, device):
    rows = []
    for model_name, checkpoint_path in best_model_paths.items():
        builder = hw.MODEL_BUILDERS[model_name]
        model = builder(model_configs[model_name]).to(device)
        hw.load_checkpoint(model, checkpoint_path, device)
        metrics = hw.evaluate_on_test(model, shared_loaders["test_loader"], device)
        rows.append({"model": model_name, **hw.summarize_metrics(metrics)})

        hw.preview_predictions(model, shared_loaders["test_loader"], class_names, device, num_samples=6)
        hw.plot_confusion_matrix_from_preds(
            metrics["y_true"],
            metrics["y_pred"],
            class_names,
            model_name.upper(),
        )

    return pd.DataFrame(rows)

# Example after all teammates finish their best models:
# best_model_paths = {
#     "mlp": project_paths["models"] / "mlp_best.pt",
#     "cnn": project_paths["models"] / "cnn_best.pt",
#     "resnet": project_paths["models"] / "resnet_best.pt",
#     "vit": project_paths["models"] / "vit_best.pt",
# }
# model_configs = {
#     "mlp": tuned_mlp_config,
#     "cnn": cnn_config,
#     "resnet": resnet_config,
#     "vit": vit_config,
# }
# final_report_df = evaluate_saved_models(best_model_paths, model_configs, loaders, class_names, device)


## Handoff Reminder

Read `README.md` before dividing work further. It records the file map, shared API, recommended ownership split, and the merge checklist.
